In [1]:
# ==========================================================
# Word2Vec Skip-gram from scratch (follows the slide pseudocode)
# Each "# %%" marks one Colab cell
# ==========================================================

# %% Cell 1: Imports and data
import numpy as np

# Sentence data: [sentence, sentiment_label (0 = negative, 1 = positive)]
data = [
    ["I love this product, it is amazing", 1],
    ["This is the worst experience ever", 0],
    ["Absolutely wonderful and highly recommended", 1],
    ["Terrible service and waste of money", 0],
    ["Really great quality and fast shipping", 1],
    ["Bad quality, do not buy this", 0],
    ["I am very happy with my purchase", 1],
    ["Horrible, completely broken on arrival", 0]
]

mystery_sentence = "This is a fantastic and beautiful item"

# C = ["cat, rat ...", ...]  -> corpus is a list of sentences
# Labels are not needed for word2vec, so we keep only the sentence text
C = []
for row in data:
    C.append(row[0])

# Extra sentence so that "mango" (ma'am's example word) is in the vocabulary
# and actually gets trained. Remove this line if you do not want it.
C.append("I love this sweet mango")

print("=" * 50)
print("Corpus has", len(C), "sentences")
print("=" * 50)


Corpus has 9 sentences


In [2]:
# %% Cell 2: Tokenize the corpus
# Lowercase, remove commas, split on spaces
tokenized_corpus = []
for sentence in C:
    clean = sentence.lower().replace(",", "")
    words = clean.split()
    tokenized_corpus.append(words)

print("=" * 50)
print("Tokenized corpus")
print("=" * 50)
for words in tokenized_corpus:
    print(words)

Tokenized corpus
['i', 'love', 'this', 'product', 'it', 'is', 'amazing']
['this', 'is', 'the', 'worst', 'experience', 'ever']
['absolutely', 'wonderful', 'and', 'highly', 'recommended']
['terrible', 'service', 'and', 'waste', 'of', 'money']
['really', 'great', 'quality', 'and', 'fast', 'shipping']
['bad', 'quality', 'do', 'not', 'buy', 'this']
['i', 'am', 'very', 'happy', 'with', 'my', 'purchase']
['horrible', 'completely', 'broken', 'on', 'arrival']
['i', 'love', 'this', 'sweet', 'mango']


In [3]:
# %% Cell 3: Step 1 - Build vocabulary V from the corpus
vocab = []
for words in tokenized_corpus:
    for w in words:
        if w not in vocab:
            vocab.append(w)

# word -> index, and index -> word
word_to_index = {}
index_to_word = {}
for i in range(len(vocab)):
    word_to_index[vocab[i]] = i
    index_to_word[i] = vocab[i]

V = len(vocab)

print("=" * 50)
print("Vocabulary size V =", V)
print("=" * 50)
print(word_to_index)
print("mango in vocabulary?", "mango" in word_to_index)



Vocabulary size V = 43
{'i': 0, 'love': 1, 'this': 2, 'product': 3, 'it': 4, 'is': 5, 'amazing': 6, 'the': 7, 'worst': 8, 'experience': 9, 'ever': 10, 'absolutely': 11, 'wonderful': 12, 'and': 13, 'highly': 14, 'recommended': 15, 'terrible': 16, 'service': 17, 'waste': 18, 'of': 19, 'money': 20, 'really': 21, 'great': 22, 'quality': 23, 'fast': 24, 'shipping': 25, 'bad': 26, 'do': 27, 'not': 28, 'buy': 29, 'am': 30, 'very': 31, 'happy': 32, 'with': 33, 'my': 34, 'purchase': 35, 'horrible': 36, 'completely': 37, 'broken': 38, 'on': 39, 'arrival': 40, 'sweet': 41, 'mango': 42}
mango in vocabulary? True


In [7]:

# %% Cell 4: Hyperparameters (the inputs c, d, eta, T of the algorithm)
c = 2          # window size
d = 10         # embedding dimension
eta = 0.05     # learning rate
T = 300        # number of epochs

np.random.seed(42)   # same random numbers every run


In [8]:

# %% Cell 5: Step 2 - Initialize input embeddings E randomly
# annotation: D = {'mango': [0.5, ...], 'cat': [ ... ]}
# E is a dictionary: word -> vector of d random numbers
E = {}
for word in vocab:
    E[word] = np.random.uniform(-0.5, 0.5, d)

print("=" * 50)
print("Initial embedding of 'mango' (random)")
print("=" * 50)
print(E["mango"])



Initial embedding of 'mango' (random)
[ 0.29157904  0.28961814 -0.4087939  -0.0055797  -0.44244124  0.04952888
 -0.0584695   0.38770418 -0.14908499 -0.38293298]


In [9]:

# %% Cell 6: Step 3 - Initialize output weights W randomly
# ma'am's annotation: W = [ ]  -> a d x V matrix
# Column j of W is the output (context) vector of word j
W = np.random.uniform(-0.5, 0.5, (d, V))

print("=" * 50)
print("Shape of W (d x V):", W.shape)
print("=" * 50)



Shape of W (d x V): (10, 43)


In [10]:

# %% Cell 7: Step 4 - Generate training pairs (center, context)
training_pairs = []

print("=" * 50)
print("Training pairs (center, context), sentence by sentence, window c =", c)
print("=" * 50)

for words in tokenized_corpus:                 # for each sentence in Corpus
    print()
    print("Sentence:", " ".join(words))
    pairs_in_sentence = 0

    for i in range(len(words)):                # for each word w at position i
        center_word = words[i]
        for j in range(i - c, i + c + 1):      # each context position j within window c
            if j < 0 or j >= len(words):       # skip positions outside the sentence
                continue
            if i != j:                         # if i != j
                training_pairs.append((center_word, words[j]))
                pairs_in_sentence = pairs_in_sentence + 1
                print("  (" + center_word + ", " + words[j] + ")")

    print("  -> pairs from this sentence:", pairs_in_sentence)

print()
print("=" * 50)
print("Total number of training pairs:", len(training_pairs))
print("=" * 50)



Training pairs (center, context), sentence by sentence, window c = 2

Sentence: i love this product it is amazing
  (i, love)
  (i, this)
  (love, i)
  (love, this)
  (love, product)
  (this, i)
  (this, love)
  (this, product)
  (this, it)
  (product, love)
  (product, this)
  (product, it)
  (product, is)
  (it, this)
  (it, product)
  (it, is)
  (it, amazing)
  (is, product)
  (is, it)
  (is, amazing)
  (amazing, it)
  (amazing, is)
  -> pairs from this sentence: 22

Sentence: this is the worst experience ever
  (this, is)
  (this, the)
  (is, this)
  (is, the)
  (is, worst)
  (the, this)
  (the, is)
  (the, worst)
  (the, experience)
  (worst, is)
  (worst, the)
  (worst, experience)
  (worst, ever)
  (experience, the)
  (experience, worst)
  (experience, ever)
  (ever, worst)
  (ever, experience)
  -> pairs from this sentence: 18

Sentence: absolutely wonderful and highly recommended
  (absolutely, wonderful)
  (absolutely, and)
  (wonderful, absolutely)
  (wonderful, and)
  (wond

In [11]:

# %% Cell 8: Softmax helper
def softmax(scores):
    # subtract the max first so exp() does not overflow (result is unchanged)
    shifted = scores - np.max(scores)
    exps = np.exp(shifted)
    return exps / np.sum(exps)



In [12]:

# %% Cell 9: Step 5 - Training loop
for epoch in range(1, T + 1):

    # Shuffle training pairs
    order = np.random.permutation(len(training_pairs))

    total_loss = 0.0

    for k in order:
        center, context = training_pairs[k]
        context_index = word_to_index[context]

        # ---- Forward pass ----
        h = E[center]                       # h = E[center]
        scores = np.dot(h, W)               # scores = h x W      (V numbers)
        probabilities = softmax(scores)     # probabilities = Softmax(scores)

        # ---- Compute loss ----
        loss = -np.log(probabilities[context_index])
        total_loss = total_loss + loss

        # ---- Backpropagation ----
        error = probabilities.copy()        # error = probabilities
        error[context_index] -= 1           # error[context] -= 1  (predicted - truth)

        gradient_W = np.outer(h, error)     # gradient_W = h^T x error   (d x V)
        gradient_h = np.dot(W, error)       # gradient_h = W x error     (d numbers)

        # ---- Update parameters ----
        W = W - eta * gradient_W
        E[center] = E[center] - eta * gradient_h

    # Print the average loss now and then
    if epoch == 1 or epoch % 50 == 0:
        print("Epoch", epoch, "| average loss =", round(total_loss / len(training_pairs), 4))



Epoch 1 | average loss = 3.8028
Epoch 50 | average loss = 1.6844
Epoch 100 | average loss = 1.6642
Epoch 150 | average loss = 1.658
Epoch 200 | average loss = 1.6592
Epoch 250 | average loss = 1.6523
Epoch 300 | average loss = 1.6547


In [13]:

# %% Cell 10: Step 6 - Return E
print("=" * 50)
print("Trained embedding of 'mango'")
print("=" * 50)
print(E["mango"])



Trained embedding of 'mango'
[ 0.55266894 -0.01382159 -0.85746134 -1.01983922 -2.51973347 -0.11612821
 -0.50802224  0.03909853 -0.70944835 -0.57750666]


In [14]:

# %% Cell 11: Cosine similarity between words
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("=" * 50)
print("Words most similar to 'love'")
print("=" * 50)
scores_list = []
for word in vocab:
    if word != "love":
        scores_list.append((cosine(E["love"], E[word]), word))
scores_list.sort(reverse=True)
for s, w in scores_list[:5]:
    print(w, round(s, 3))



Words most similar to 'love'
mango 0.727
it 0.592
buy 0.539
am 0.537
is 0.437


In [15]:

# %% Cell 12: Mystery sentence - which words have a trained vector?
print("=" * 50)
print("Mystery sentence check")
print("=" * 50)
for w in mystery_sentence.lower().split():
    if w in E:
        print(w, "-> has a vector")
    else:
        print(w, "-> NOT in vocabulary (never seen in training)")

Mystery sentence check
this -> has a vector
is -> has a vector
a -> NOT in vocabulary (never seen in training)
fantastic -> NOT in vocabulary (never seen in training)
and -> has a vector
beautiful -> NOT in vocabulary (never seen in training)
item -> NOT in vocabulary (never seen in training)
